# Multi-Agent DevOps Incident Analysis Suite — Clean Contract Notebook v5

This notebook is intentionally smaller than v3/v4.

It keeps the good implementation knowledge from earlier notebooks, but exposes only a few stable entry points:

1. `analyze_uploaded_logs(files_or_paths, options=None)` — UI/FastAPI entry point  
2. `run_analysis(request)` — backend pipeline entry point  
3. `retrieve_context(analysis_context)` — RAG boundary  
4. `run_incident_graph(graph_input)` — LangGraph/LLM reasoning boundary  
5. `build_action_payload(analysis_response, approval_status="PREVIEW_ONLY")` — n8n/JIRA/Slack boundary  

Internal helper functions can later move into `src/ingest`, `src/signals`, `src/clustering`, `src/rag`, `src/graph`, and `src/reporting`.

## 0. Install dependencies

This notebook uses lightweight dependencies. LangGraph is optional in this v5 notebook because the graph is kept intentionally simple for onboarding.

In [ ]:
!pip -q install pandas pydantic requests python-dateutil
# Optional later:
# !pip -q install langgraph langchain langchain-openai

## 1. Configuration

Uses `OPENROUTER_API_KEY` first.

In Colab:
```python
import os
os.environ["OPENROUTER_API_KEY"] = "your-key"
```

The code uses OpenRouter-compatible chat completions:
- Base URL: `https://openrouter.ai/api/v1`
- Model: configurable through `MODEL_NAME`

In [ ]:
import os, json, re, csv, zipfile, shutil, hashlib, textwrap, requests
from pathlib import Path
from datetime import datetime
from dateutil import parser as dtparser
from typing import Any, Dict, List, Optional, Tuple
from pydantic import BaseModel, Field

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
MODEL_NAME = os.getenv("MODEL_NAME", "openai/gpt-4o-mini")

DEFAULT_OPTIONS = {
    "enable_rag": True,
    "enable_llm": bool(OPENROUTER_API_KEY),
    "preview_only": True,
    "run_evals": False,
    "max_clusters": 12,
    "max_evidence_per_cluster": 12
}

WORK_ROOT = Path("/content/incident_suite_v5") if Path("/content").exists() else Path("/mnt/data/incident_suite_v5")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

## 2. JSON/Object Contracts

These are the shared contracts team members should code against. They are intentionally compact.

In [ ]:
class AnalysisRequest(BaseModel):
    run_id: str
    uploaded_paths: List[str]
    options: Dict[str, Any] = Field(default_factory=dict)

class LogEvent(BaseModel):
    source_file: str
    line_no: int
    timestamp: Optional[str] = None
    level: str = "UNKNOWN"
    service: str = "unknown"
    message: str
    trace_id: Optional[str] = None
    raw_line: str
    parse_status: str = "parsed"
    metadata: Dict[str, Any] = Field(default_factory=dict)

class SignalEvidence(BaseModel):
    source_file: str
    line_no: int
    timestamp: Optional[str] = None
    service: str = "unknown"
    level: str = "UNKNOWN"
    message: str
    trace_id: Optional[str] = None

class SignalMatch(BaseModel):
    signal_type: str
    rule_id: str
    weight: float
    severity_hint: str = "UNKNOWN"
    evidence: SignalEvidence

class EvidenceCluster(BaseModel):
    cluster_id: str
    candidate_category: str
    signature: str
    affected_services: List[str]
    start_ts: Optional[str] = None
    end_ts: Optional[str] = None
    signal_count: int
    total_weight: float
    severity_hint: str = "UNKNOWN"
    evidence: List[SignalEvidence]
    rule_ids: List[str] = Field(default_factory=list)
    summary: str = ""

class RAGChunk(BaseModel):
    title: str
    incident_type: str
    symptoms: List[str]
    diagnostics: List[str]
    remediation: List[str]
    validation: List[str]
    safety_notes: List[str]
    owner: str = "sre"
    source: str = "embedded_runbook"
    score: float = 0.0

class RAGResult(BaseModel):
    run_id: str
    rag_results: List[Dict[str, Any]]

class IncidentResult(BaseModel):
    incident_id: str
    cluster_id: str
    category: str
    severity: str
    confidence: float
    title: str
    affected_services: List[str]
    symptom_vs_cause: str
    timeline: Dict[str, Any]
    root_cause: Dict[str, Any]
    remediation: Dict[str, Any]
    critic_findings: List[str] = Field(default_factory=list)
    evidence_grounded: bool = True

class TicketPayload(BaseModel):
    system: str = "jira"
    create_issue: bool = True
    project_key: str = "OPS"
    title: str
    priority: str
    labels: List[str]
    description: str
    evidence: List[str]
    approval_status: str

class SlackPayload(BaseModel):
    channel: str = "#devops-incidents"
    send_message: bool = True
    title: str
    body: str
    approval_status: str

class CookbookPayload(BaseModel):
    title: str
    steps: List[str]
    validation: List[str]
    safety_notes: List[str]

class ActionPayload(BaseModel):
    run_id: str
    approval_status: str
    incident_id: str
    severity: str
    category: str
    ticket: TicketPayload
    slack: SlackPayload
    cookbook: CookbookPayload

class AnalysisResponse(BaseModel):
    run_id: str
    status: str
    summary: Dict[str, Any]
    clusters: List[EvidenceCluster]
    rag_results: List[Dict[str, Any]]
    incidents: List[IncidentResult]
    action_payloads: List[ActionPayload]
    exports: Dict[str, str] = Field(default_factory=dict)

## 3. Public Entry Point 1 — `analyze_uploaded_logs`

This is the only function Gradio/FastAPI needs to call.

In [ ]:
def new_run_id() -> str:
    return "run_" + datetime.utcnow().strftime("%Y%m%d_%H%M%S")

def analyze_uploaded_logs(files_or_paths: List[Any], options: Optional[Dict[str, Any]] = None) -> AnalysisResponse:
    opts = {**DEFAULT_OPTIONS, **(options or {})}
    run_id = new_run_id()

    uploaded_paths = []
    run_upload_dir = WORK_ROOT / "uploads" / run_id
    run_upload_dir.mkdir(parents=True, exist_ok=True)

    for item in files_or_paths:
        # Supports paths, Colab upload dict values, Gradio temp files, or file-like objects.
        if isinstance(item, (str, Path)):
            src = Path(item)
            dst = run_upload_dir / src.name
            if src.resolve() != dst.resolve():
                shutil.copy(src, dst)
            uploaded_paths.append(str(dst))
        elif hasattr(item, "name") and Path(item.name).exists():
            src = Path(item.name)
            dst = run_upload_dir / src.name
            shutil.copy(src, dst)
            uploaded_paths.append(str(dst))
        else:
            raise ValueError(f"Unsupported upload object: {type(item)}")

    request = AnalysisRequest(run_id=run_id, uploaded_paths=uploaded_paths, options=opts)
    return run_analysis(request)

## 4. Public Entry Point 2 — `run_analysis`

This function internally runs ingestion, signal extraction, clustering, RAG, LangGraph-style reasoning, action building, and exports.

In [ ]:
def run_analysis(request: AnalysisRequest) -> AnalysisResponse:
    run_root = WORK_ROOT / "runs" / request.run_id
    input_root = run_root / "input"
    output_root = run_root / "outputs"
    input_root.mkdir(parents=True, exist_ok=True)
    output_root.mkdir(parents=True, exist_ok=True)

    raw_files, ground_truth_files = prepare_inputs(request.uploaded_paths, input_root)

    events = []
    errors = []
    for path in raw_files:
        try:
            events.extend(parse_log_file(path, input_root))
        except Exception as e:
            errors.append({"file": str(path), "error": str(e)})

    signals = extract_signals(events)
    clusters = build_evidence_clusters(
        signals,
        max_evidence_per_cluster=request.options.get("max_evidence_per_cluster", 12),
        max_clusters=request.options.get("max_clusters", 12)
    )

    analysis_context = {
        "run_id": request.run_id,
        "clusters": [c.model_dump() for c in clusters]
    }

    rag_result = retrieve_context(analysis_context) if request.options.get("enable_rag", True) else {"run_id": request.run_id, "rag_results": []}

    graph_input = {
        "run_id": request.run_id,
        "clusters": [c.model_dump() for c in clusters],
        "rag_results": rag_result.get("rag_results", []),
        "options": request.options
    }
    graph_output = run_incident_graph(graph_input)
    incidents = [IncidentResult(**x) for x in graph_output["incidents"]]

    action_payloads = [
        build_action_payload(request.run_id, incident, approval_status="PREVIEW_ONLY" if request.options.get("preview_only", True) else "PENDING_HUMAN_APPROVAL")
        for incident in incidents
    ]

    exports = export_results(output_root, request.run_id, events, clusters, rag_result.get("rag_results", []), incidents, action_payloads)

    return AnalysisResponse(
        run_id=request.run_id,
        status="completed",
        summary={
            "raw_files": len(raw_files),
            "ground_truth_files_detected": len(ground_truth_files),
            "events_parsed": len(events),
            "signals_found": len(signals),
            "clusters_found": len(clusters),
            "incidents_found": len(incidents),
            "errors": errors
        },
        clusters=clusters,
        rag_results=rag_result.get("rag_results", []),
        incidents=incidents,
        action_payloads=action_payloads,
        exports=exports
    )

## 5. Ingestion and Parsing Internals

Later refactor target:
- `src/ingest/input_validator.py`
- `src/ingest/zip_loader.py`
- `src/ingest/log_parser.py`
- `src/ingest/normalizer.py`

In [ ]:
ALLOWED_EXTENSIONS = {".jsonl", ".log", ".txt"}

def safe_extract_zip(zip_path: Path, dest: Path) -> None:
    with zipfile.ZipFile(zip_path, "r") as z:
        for member in z.infolist():
            target = dest / member.filename
            if not str(target.resolve()).startswith(str(dest.resolve())):
                raise ValueError(f"Unsafe zip path detected: {member.filename}")
        z.extractall(dest)

def prepare_inputs(uploaded_paths: List[str], input_root: Path) -> Tuple[List[Path], List[Path]]:
    for p in uploaded_paths:
        src = Path(p)
        if src.suffix.lower() == ".zip":
            safe_extract_zip(src, input_root)
        elif src.suffix.lower() in ALLOWED_EXTENSIONS:
            shutil.copy(src, input_root / src.name)
        else:
            raise ValueError(f"Unsupported file extension: {src.suffix}")

    raw_files, ground_truth_files = [], []
    for path in input_root.rglob("*"):
        if not path.is_file():
            continue
        rel = str(path.relative_to(input_root))
        if "ground_truth_eval_only" in rel:
            ground_truth_files.append(path)
        elif path.suffix.lower() in ALLOWED_EXTENSIONS:
            raw_files.append(path)
    return sorted(raw_files), sorted(ground_truth_files)

def normalize_level(value: Any, message: str = "") -> str:
    text = str(value or "").upper()
    if text in {"ERROR", "ERR", "FATAL", "CRITICAL"}: return "ERROR"
    if text in {"WARN", "WARNING"}: return "WARN"
    if text in {"INFO", "DEBUG", "TRACE"}: return text
    msg = message.upper()
    if any(x in msg for x in ["ERROR", "EXCEPTION", "FAILED", "TIMEOUT", "OOMKILLED"]): return "ERROR"
    if any(x in msg for x in ["WARN", "RETRY", "DEGRADED"]): return "WARN"
    return "INFO"

def parse_ts(value: Any) -> Optional[str]:
    if not value:
        return None
    try:
        return dtparser.parse(str(value)).isoformat()
    except Exception:
        return None

def parse_json_or_text_line(line: str, source_file: str, line_no: int) -> LogEvent:
    raw = line.rstrip("\n")
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict):
            message = str(obj.get("message") or obj.get("msg") or obj.get("log") or obj.get("event") or raw)
            service = str(obj.get("service") or obj.get("app") or obj.get("logger") or obj.get("component") or "unknown")
            timestamp = parse_ts(obj.get("timestamp") or obj.get("ts") or obj.get("time") or obj.get("@timestamp"))
            level = normalize_level(obj.get("level") or obj.get("severity"), message)
            trace_id = obj.get("trace_id") or obj.get("traceId") or obj.get("correlation_id")
            return LogEvent(
                source_file=source_file, line_no=line_no, timestamp=timestamp, level=level,
                service=service, message=message, trace_id=trace_id, raw_line=raw,
                metadata={k:v for k,v in obj.items() if k not in {"message","msg","log","event"}}
            )
    except Exception:
        pass

    # Plain text fallback
    timestamp = None
    m = re.search(r"(\d{4}-\d{2}-\d{2}[T\s]\d{2}:\d{2}:\d{2}(?:\.\d+)?Z?)", raw)
    if m: timestamp = parse_ts(m.group(1))
    level = normalize_level("", raw)
    svc = "unknown"
    m2 = re.search(r"(service|app|component)=([A-Za-z0-9_.-]+)", raw)
    if m2: svc = m2.group(2)
    return LogEvent(source_file=source_file, line_no=line_no, timestamp=timestamp, level=level, service=svc, message=raw, raw_line=raw)

def parse_log_file(path: Path, input_root: Path) -> List[LogEvent]:
    rel = str(path.relative_to(input_root))
    events = []
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f, start=1):
            if line.strip():
                events.append(parse_json_or_text_line(line, rel, i))
    return events

## 6. Signal Extraction and Clustering Internals

Later refactor target:
- `src/signals/rules.py`
- `src/signals/signal_extractor.py`
- `src/clustering/evidence_clusterer.py`

In [ ]:
SIGNAL_RULES = [
    ("Database", "db_hikari_timeout", 4.0, "P1", r"hikaripool|connection pool|too many clients|lock wait|deadlock|slow query|pg_stat_activity"),
    ("Network", "network_connectivity", 3.0, "P2", r"dns failure|connection reset|connection refused|tls handshake|tcp retransmit|network unreachable"),
    ("Authentication", "auth_failure", 3.0, "P2", r"jwt|invalid signature|issuer mismatch|jwk|401|403|rbac|unauthorized|forbidden"),
    ("Memory/CPU", "resource_pressure", 4.0, "P1", r"oomkilled|out of memory|heap|gc pause|cpu throttl|restart loop"),
    ("Deployment regression", "deployment_regression", 4.0, "P1", r"rollout|canary failed|rollback|feature flag|configmap|new version|deployment"),
    ("API timeout", "api_timeout", 3.0, "P2", r"504|gateway timeout|upstream timed out|p95|latency breach|circuit breaker"),
    ("Queue backlog", "queue_backlog", 3.0, "P2", r"consumer lag|dlq|dead letter|retry topic|poison message|rebalance"),
    ("Disk/storage", "disk_storage", 4.0, "P1", r"no space left|disk pressure|logrotate failed|io wait|persistent volume|storage latency"),
    ("External dependency", "external_dependency", 3.0, "P2", r"vendor|provider 5xx|http 429|retry-after|third-party|external dependency"),
    ("Unknown", "unknown_insufficient_evidence", 1.0, "P3", r"missing correlation|incomplete trace|redacted|unexpected state|unknown error")
]

def evidence_from_event(e: LogEvent) -> SignalEvidence:
    return SignalEvidence(
        source_file=e.source_file, line_no=e.line_no, timestamp=e.timestamp,
        service=e.service, level=e.level, message=e.message, trace_id=e.trace_id
    )

def extract_signals(events: List[LogEvent]) -> List[SignalMatch]:
    matches = []
    for e in events:
        text = f"{e.level} {e.service} {e.message}".lower()
        for category, rule_id, weight, sev, pattern in SIGNAL_RULES:
            if re.search(pattern, text, re.I):
                matches.append(SignalMatch(
                    signal_type=category,
                    rule_id=rule_id,
                    weight=weight,
                    severity_hint=sev,
                    evidence=evidence_from_event(e)
                ))
    return matches

def severity_from_weight(total_weight: float, hints: List[str]) -> str:
    if "P1" in hints or total_weight >= 25: return "P1"
    if "P2" in hints or total_weight >= 10: return "P2"
    if total_weight >= 3: return "P3"
    return "UNKNOWN"

def stable_id(text: str) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:10]

def build_evidence_clusters(signals: List[SignalMatch], max_evidence_per_cluster=12, max_clusters=12) -> List[EvidenceCluster]:
    grouped: Dict[Tuple[str, str], List[SignalMatch]] = {}
    for s in signals:
        service = s.evidence.service or "unknown"
        key = (s.signal_type, service)
        grouped.setdefault(key, []).append(s)

    clusters = []
    for (category, service), items in grouped.items():
        items = sorted(items, key=lambda x: (x.evidence.timestamp or "", x.evidence.source_file, x.evidence.line_no))
        total_weight = sum(x.weight for x in items)
        hints = [x.severity_hint for x in items]
        evidence = [x.evidence for x in items[:max_evidence_per_cluster]]
        start_ts = next((x.evidence.timestamp for x in items if x.evidence.timestamp), None)
        end_ts = next((x.evidence.timestamp for x in reversed(items) if x.evidence.timestamp), None)
        cid = f"cluster_{stable_id(category + service + str(len(items)))}"
        clusters.append(EvidenceCluster(
            cluster_id=cid,
            candidate_category=category,
            signature=items[0].rule_id,
            affected_services=sorted({x.evidence.service for x in items if x.evidence.service}),
            start_ts=start_ts,
            end_ts=end_ts,
            signal_count=len(items),
            total_weight=round(total_weight, 2),
            severity_hint=severity_from_weight(total_weight, hints),
            evidence=evidence,
            rule_ids=sorted({x.rule_id for x in items}),
            summary=f"{len(items)} {category} signal(s) detected for service {service}."
        ))

    clusters.sort(key=lambda c: (c.severity_hint != "P1", -c.total_weight))
    return clusters[:max_clusters]

## 7. Public Entry Point 3 — `retrieve_context`

This is Pawan's clean RAG boundary.  
Input: clusters.  
Output: runbook matches by cluster.

In [ ]:
RUNBOOKS = [
    RAGChunk(
        title="Database connection pool exhaustion",
        incident_type="Database",
        symptoms=["HikariPool", "too many clients", "lock wait", "slow query"],
        diagnostics=["Check pg_stat_activity", "Check DB active sessions", "Inspect lock waits", "Review connection pool metrics"],
        remediation=["Identify long-running transactions", "Reduce request pressure", "Tune pool only after DB capacity check"],
        validation=["DB wait time returns to baseline", "5xx rate decreases", "Pool acquisition latency normalizes"],
        safety_notes=["Do not restart DB primary blindly", "Avoid increasing pool size without DB capacity validation"],
    ),
    RAGChunk(
        title="API timeout cascade",
        incident_type="API timeout",
        symptoms=["504", "gateway timeout", "upstream timed out", "circuit breaker"],
        diagnostics=["Check downstream latency", "Review p95/p99 latency", "Inspect circuit breaker state"],
        remediation=["Reduce timeout cascade", "Scale bottleneck service if safe", "Apply backpressure"],
        validation=["Gateway 504 rate decreases", "p95 latency returns to baseline"],
        safety_notes=["Do not disable circuit breakers without approval"],
    ),
    RAGChunk(
        title="Authentication token validation issue",
        incident_type="Authentication",
        symptoms=["JWT", "invalid signature", "JWK", "401", "403"],
        diagnostics=["Check issuer/audience config", "Verify JWK refresh", "Review recent auth config changes"],
        remediation=["Refresh key cache", "Rollback bad auth config", "Validate token issuer settings"],
        validation=["401/403 rate normalizes", "Valid tokens succeed"],
        safety_notes=["Do not weaken token validation rules"],
    ),
    RAGChunk(
        title="Queue backlog and DLQ growth",
        incident_type="Queue backlog",
        symptoms=["consumer lag", "DLQ", "poison message", "retry topic"],
        diagnostics=["Check consumer lag", "Inspect DLQ samples", "Identify poison messages"],
        remediation=["Pause poison message replay", "Scale consumers if downstream is healthy", "Fix retry storm"],
        validation=["Consumer lag decreases", "DLQ growth stops"],
        safety_notes=["Do not blindly replay DLQ into production"],
    ),
    RAGChunk(
        title="Resource pressure: memory or CPU saturation",
        incident_type="Memory/CPU",
        symptoms=["OOMKilled", "heap", "GC pause", "CPU throttling"],
        diagnostics=["Check pod restarts", "Review heap and CPU metrics", "Inspect recent traffic or deployment changes"],
        remediation=["Rollback memory regression", "Increase limits cautiously", "Reduce traffic if needed"],
        validation=["Restart count stabilizes", "GC pause and CPU throttling reduce"],
        safety_notes=["Human approval before disruptive restarts"],
    ),
    RAGChunk(
        title="External dependency degradation",
        incident_type="External dependency",
        symptoms=["vendor", "provider 5xx", "HTTP 429", "Retry-After"],
        diagnostics=["Check vendor status page", "Inspect retry volume", "Confirm throttling headers"],
        remediation=["Enable fallback path", "Throttle retries", "Use cached response where safe"],
        validation=["Vendor error rate decreases", "Retry volume normalizes"],
        safety_notes=["Avoid retry storms against degraded vendors"],
    )
]

def retrieve_context(analysis_context: Dict[str, Any]) -> Dict[str, Any]:
    results = []
    for cluster in analysis_context.get("clusters", []):
        category = cluster.get("candidate_category", "")
        haystack = " ".join([
            category,
            cluster.get("signature", ""),
            " ".join(cluster.get("affected_services", [])),
            " ".join([e.get("message", "") for e in cluster.get("evidence", [])])
        ]).lower()

        scored = []
        for rb in RUNBOOKS:
            score = 0
            if rb.incident_type.lower() == category.lower():
                score += 5
            for term in rb.symptoms + rb.diagnostics:
                if term.lower() in haystack:
                    score += 1
            if score > 0:
                item = rb.model_dump()
                item["score"] = round(score / 10, 2)
                scored.append(item)

        scored.sort(key=lambda x: x["score"], reverse=True)
        results.append({
            "cluster_id": cluster.get("cluster_id"),
            "query": f"{category} {cluster.get('signature')} {' '.join(cluster.get('affected_services', []))}",
            "chunks": scored[:3]
        })

    return {"run_id": analysis_context.get("run_id"), "rag_results": results}

## 8. Public Entry Point 4 — `run_incident_graph`

This is intentionally simpler than v3/v4. It behaves like a compact LangGraph boundary:

```text
cluster + RAG context
    -> classify/severity
    -> RCA
    -> remediation
    -> critic
    -> incident result
```

If `OPENROUTER_API_KEY` exists, it uses OpenRouter. Otherwise it uses deterministic fallback.

In [ ]:
def openrouter_chat_json(system_prompt: str, user_payload: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    if not OPENROUTER_API_KEY:
        return None
    try:
        resp = requests.post(
            f"{OPENROUTER_BASE_URL}/chat/completions",
            headers={
                "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                "Content-Type": "application/json",
                "HTTP-Referer": "https://colab.research.google.com/",
                "X-Title": "Multi-Agent DevOps Incident Analysis Suite v5"
            },
            json={
                "model": MODEL_NAME,
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": json.dumps(user_payload, ensure_ascii=False)}
                ],
                "temperature": 0.1,
                "response_format": {"type": "json_object"}
            },
            timeout=60
        )
        resp.raise_for_status()
        content = resp.json()["choices"][0]["message"]["content"]
        return json.loads(content)
    except Exception as e:
        print("LLM fallback used:", e)
        return None

def evidence_ref(e: Dict[str, Any]) -> str:
    return f"{e.get('source_file')}:{e.get('line_no')} - {e.get('message')}"

def fallback_incident(cluster: Dict[str, Any], rag_chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
    category = cluster.get("candidate_category", "Unknown")
    severity = cluster.get("severity_hint", "P3")
    evidence = cluster.get("evidence", [])
    top_refs = [evidence_ref(e) for e in evidence[:5]]
    rb = rag_chunks[0] if rag_chunks else {}

    title = f"{severity}: {category} incident candidate"
    root = {
        "primary_root_cause": f"Likely {category} issue based on matching operational signals.",
        "alternative_causes": ["Deployment/config change", "External dependency", "Insufficient telemetry"],
        "supporting_evidence": top_refs,
        "missing_evidence": ["Metrics", "Traces", "Recent deployment history"],
        "confidence": 0.72 if evidence else 0.4
    }
    remediation = {
        "immediate_actions": rb.get("diagnostics", ["Review evidence lines", "Check service health", "Validate recent changes"]),
        "validation_checks": rb.get("validation", ["Error rate decreases", "Latency returns to baseline"]),
        "safety_notes": rb.get("safety_notes", ["Human approval recommended before disruptive actions"]),
        "requires_human_approval": True
    }
    timeline = {
        "events": evidence[:8],
        "interpretation": f"{category} signals were observed across {cluster.get('affected_services', [])}."
    }
    return {
        "incident_id": cluster.get("cluster_id"),
        "cluster_id": cluster.get("cluster_id"),
        "category": category,
        "severity": severity,
        "confidence": root["confidence"],
        "title": title,
        "affected_services": cluster.get("affected_services", []),
        "symptom_vs_cause": f"{category} is the strongest candidate from deterministic signals. Related timeout/error lines may be symptoms.",
        "timeline": timeline,
        "root_cause": root,
        "remediation": remediation,
        "critic_findings": ["Fallback mode: evidence-grounded preview generated. LLM review can improve RCA detail."],
        "evidence_grounded": bool(evidence)
    }

def run_incident_graph(graph_input: Dict[str, Any]) -> Dict[str, Any]:
    rag_by_cluster = {
        r.get("cluster_id"): r.get("chunks", [])
        for r in graph_input.get("rag_results", [])
    }
    incidents = []
    use_llm = graph_input.get("options", {}).get("enable_llm", False)

    for cluster in graph_input.get("clusters", []):
        rag_chunks = rag_by_cluster.get(cluster.get("cluster_id"), [])

        llm_result = None
        if use_llm and OPENROUTER_API_KEY:
            system_prompt = """
You are an SRE incident analysis agent. Return ONLY valid JSON matching:
{
  "category": "...",
  "severity": "P1|P2|P3|P4|UNKNOWN",
  "confidence": 0.0,
  "title": "...",
  "symptom_vs_cause": "...",
  "root_cause": {
    "primary_root_cause": "...",
    "alternative_causes": [],
    "supporting_evidence": [],
    "missing_evidence": [],
    "confidence": 0.0
  },
  "remediation": {
    "immediate_actions": [],
    "validation_checks": [],
    "safety_notes": [],
    "requires_human_approval": true
  },
  "critic_findings": []
}
Rules:
- Use only provided evidence and RAG context.
- Cite evidence as file:line - message.
- Do not read or infer hidden ground truth.
- If evidence is weak, say so.
- Recommend safe actions only.
"""
            llm_result = openrouter_chat_json(system_prompt, {"cluster": cluster, "rag_chunks": rag_chunks})

        base = fallback_incident(cluster, rag_chunks)
        if llm_result:
            base.update({
                "category": llm_result.get("category", base["category"]),
                "severity": llm_result.get("severity", base["severity"]),
                "confidence": llm_result.get("confidence", base["confidence"]),
                "title": llm_result.get("title", base["title"]),
                "symptom_vs_cause": llm_result.get("symptom_vs_cause", base["symptom_vs_cause"]),
                "root_cause": llm_result.get("root_cause", base["root_cause"]),
                "remediation": llm_result.get("remediation", base["remediation"]),
                "critic_findings": llm_result.get("critic_findings", ["LLM review completed."])
            })
        incidents.append(base)

    return {"run_id": graph_input.get("run_id"), "incidents": incidents}

## 9. Public Entry Point 5 — `build_action_payload`

This is Rajesh's n8n boundary.  
By default it remains preview-only unless approval is explicitly changed.

In [ ]:
def build_action_payload(run_id: str, incident: IncidentResult, approval_status: str = "PREVIEW_ONLY") -> ActionPayload:
    evidence = incident.root_cause.get("supporting_evidence", [])
    actions = incident.remediation.get("immediate_actions", [])
    validation = incident.remediation.get("validation_checks", [])
    safety = incident.remediation.get("safety_notes", [])

    ticket = TicketPayload(
        title=f"{incident.severity}: {incident.category} - {incident.title}",
        priority=incident.severity,
        labels=["ai-generated", "incident", incident.severity.lower(), incident.category.lower().replace("/", "-").replace(" ", "-")],
        description=(
            f"Root cause: {incident.root_cause.get('primary_root_cause')}\n\n"
            f"Evidence:\n- " + "\n- ".join(evidence[:10]) + "\n\n"
            f"Recommended actions:\n- " + "\n- ".join(actions[:10])
        ),
        evidence=evidence,
        approval_status=approval_status
    )

    slack = SlackPayload(
        title=f"{incident.severity}: {incident.category} incident candidate",
        body=(
            f"Affected services: {', '.join(incident.affected_services)}\n"
            f"Likely cause: {incident.root_cause.get('primary_root_cause')}\n"
            f"Top action: {(actions or ['Review evidence'])[0]}\n"
            f"Status: {approval_status}"
        ),
        approval_status=approval_status
    )

    cookbook = CookbookPayload(
        title=f"Cookbook: {incident.category} / {incident.cluster_id}",
        steps=actions,
        validation=validation,
        safety_notes=safety
    )

    return ActionPayload(
        run_id=run_id,
        approval_status=approval_status,
        incident_id=incident.incident_id,
        severity=incident.severity,
        category=incident.category,
        ticket=ticket,
        slack=slack,
        cookbook=cookbook
    )

## 10. Reporting and Exports

Later refactor target:
- `src/reporting/json_exporter.py`
- `src/reporting/csv_exporter.py`
- `src/reporting/markdown_report.py`
- `src/reporting/export_bundle.py`

In [ ]:
def export_results(output_root: Path, run_id: str, events, clusters, rag_results, incidents, action_payloads) -> Dict[str, str]:
    output_root.mkdir(parents=True, exist_ok=True)

    json_path = output_root / "analysis_response.json"
    md_path = output_root / "incident_report.md"
    csv_path = output_root / "incidents.csv"

    payload = {
        "run_id": run_id,
        "summary": {
            "events_parsed": len(events),
            "clusters_found": len(clusters),
            "incidents_found": len(incidents)
        },
        "clusters": [c.model_dump() for c in clusters],
        "rag_results": rag_results,
        "incidents": [i.model_dump() for i in incidents],
        "action_payloads": [a.model_dump() for a in action_payloads]
    }
    json_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

    lines = [f"# Incident Analysis Report — {run_id}", ""]
    lines.append(f"- Events parsed: {len(events)}")
    lines.append(f"- Clusters found: {len(clusters)}")
    lines.append(f"- Incidents found: {len(incidents)}")
    lines.append("")
    for inc in incidents:
        lines.append(f"## {inc.title}")
        lines.append(f"- Severity: {inc.severity}")
        lines.append(f"- Category: {inc.category}")
        lines.append(f"- Confidence: {inc.confidence}")
        lines.append(f"- Affected services: {', '.join(inc.affected_services)}")
        lines.append(f"- Root cause: {inc.root_cause.get('primary_root_cause')}")
        lines.append("")
        lines.append("### Evidence")
        for ev in inc.root_cause.get("supporting_evidence", [])[:8]:
            lines.append(f"- {ev}")
        lines.append("")
        lines.append("### Immediate Actions")
        for step in inc.remediation.get("immediate_actions", []):
            lines.append(f"- {step}")
        lines.append("")
    md_path.write_text("\n".join(lines), encoding="utf-8")

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["incident_id", "severity", "category", "confidence", "title", "affected_services"])
        writer.writeheader()
        for inc in incidents:
            writer.writerow({
                "incident_id": inc.incident_id,
                "severity": inc.severity,
                "category": inc.category,
                "confidence": inc.confidence,
                "title": inc.title,
                "affected_services": ",".join(inc.affected_services)
            })

    return {
        "json_export": str(json_path),
        "markdown_report": str(md_path),
        "csv_export": str(csv_path)
    }

def show_response_summary(response: AnalysisResponse):
    print(json.dumps(response.summary, indent=2))
    for inc in response.incidents[:10]:
        print(f"\n[{inc.severity}] {inc.category}: {inc.title}")
        print(f"Services: {', '.join(inc.affected_services)}")
        print(f"Root cause: {inc.root_cause.get('primary_root_cause')}")

## 11. Run the Notebook

### Option A: Use Colab upload
```python
from google.colab import files
uploaded = files.upload()
response = analyze_uploaded_logs(list(uploaded.keys()))
show_response_summary(response)
```

### Option B: Use local path
```python
response = analyze_uploaded_logs(["/path/to/logs.zip"])
show_response_summary(response)
```

In [ ]:
# Example usage:
# response = analyze_uploaded_logs(["/mnt/data/devops_raw_log_corpus_v2_all.zip"], options={"enable_llm": False, "enable_rag": True})
# show_response_summary(response)
# response.exports

## 12. Refactor Map from Notebook v5 to `src/`

Once v5 is stable, split it like this:

| v5 notebook function/class | Future file |
|---|---|
| `AnalysisRequest`, `AnalysisResponse`, `LogEvent`, `EvidenceCluster`, `IncidentResult`, payload classes | `src/models/*.py` |
| `analyze_uploaded_logs` | `src/api` or `src/app` entrypoint |
| `run_analysis` | `src/graph/runner.py` or `src/pipeline.py` |
| `prepare_inputs`, `parse_log_file` | `src/ingest/*.py` |
| `extract_signals` | `src/signals/signal_extractor.py` |
| `build_evidence_clusters` | `src/clustering/evidence_clusterer.py` |
| `retrieve_context` | `src/rag/retriever.py` |
| `run_incident_graph` | `src/graph/runner.py` |
| `build_action_payload` | `src/reporting/action_builder.py` or `src/adapters/payload_builder.py` |
| `export_results` | `src/reporting/*.py` |